In this ection we will discuss depth vs breadth 

1. Depth is the number of hidden layers in a meural network  

2. Breadth is the number of total perceptrons in a hidden layers.


In [20]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import torch 
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_iris
iris = load_iris()
X = torch.tensor(iris.data, dtype=torch.float32)
y = torch.tensor(iris.target, dtype=torch.long)

In [21]:
class ANNiris(nn.Module):
  def __init__(self,nUnits,nLayers):
    super().__init__()

    # create dictionary to store the layers
    self.layers = nn.ModuleDict()
    self.nLayers = nLayers #nLayers

    # input layer
    self.layers['input'] = nn.Linear(4,nUnits)

    # hidden layers
    for i in range(nLayers):
      self.layers[f'hidden{i}'] = nn.Linear(nUnits,nUnits)

    # output layer
    self.layers['output'] = nn.Linear(nUnits,3)
        
      # forward pass
  def forward(self,x):
    # input layer (note: the code in the video omits the relu after this layer)
    x = F.relu(self.layers['input'](x))

    # hidden layers
    for i in range(self.nLayers):
      x = F.relu( self.layers[f'hidden{i}'](x) )

    # return output layer
    x = self.layers['output'](x)
    return x



In [22]:
# generate an instance of the model and inspect it
nUnitsPerLayer = 12
nLayers = 4
net = ANNiris(nUnitsPerLayer,nLayers)
net


ANNiris(
  (layers): ModuleDict(
    (input): Linear(in_features=4, out_features=12, bias=True)
    (hidden0): Linear(in_features=12, out_features=12, bias=True)
    (hidden1): Linear(in_features=12, out_features=12, bias=True)
    (hidden2): Linear(in_features=12, out_features=12, bias=True)
    (hidden3): Linear(in_features=12, out_features=12, bias=True)
    (output): Linear(in_features=12, out_features=3, bias=True)
  )
)

In [32]:
width=100
depth=10
width_model=ANNiris(1 , width)
depth_model=ANNiris(depth , 1)

In [34]:
# a function to train the models 

def trainTheModel(theModel):

  # define the loss function and optimizer
  lossfun = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(theModel.parameters(),lr=.01)
  numepochs=2000
  # loop over epochs
  for epochi in range(numepochs):

    # forward pass
    yHat = theModel(X)

    # compute loss
    loss = lossfun(yHat,y)

    # backprop
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    
     # final forward pass to get accuracy
  predictions = theModel(X)
  predlabels = torch.argmax(predictions,axis=1)
  acc = 100*torch.mean((predlabels == y).float())

  # total number of trainable parameters in the model
  nParams = sum(p.numel() for p in theModel.parameters() if p.requires_grad)

  # function outputs
  return acc,nParams



In [35]:
acc,nParams=trainTheModel(width_model)
print(acc)
print(nParams)


tensor(33.3333)
211


In [36]:
acc,nParams=trainTheModel(depth_model)
print(acc)
print(nParams)


tensor(98.6667)
193
